# 4.1b Observability for the GenAI claims assistant and its agent

**Story:** The SafeDrive claims assistant is in production. The team needs to see every model call,
notice when the agent's tool fails or an answer is cut off, and get an alert when answer quality drops.

**Task 4.1:** Monitor ML and AI model inference and performance (GenAI part), plus the agent
troubleshooting tools of Task 4.2.

| Skill | What you do here | Status |
|---|---|---|
| 4.1.1 | Bedrock invocation logging, runtime metrics, CloudWatch generative AI observability | ✅ |
| 4.1.5 | Agent traces, tool failures, truncated output, loops, automated responses | ✅ |
| 4.2.2 | Amazon Bedrock AgentCore Observability and AWS X-Ray (agent part) | ✅ |
| 4.1.6 | Recurring FM quality check with a custom metric and alarm; Bedrock evaluation job 🔒 | ✅ |

## Key points for the exam

- **Model invocation logging** is off by default. One setting per account and Region. It sends the full
  request, response and token counts to **CloudWatch Logs and/or S3**.
- **Bedrock runtime metrics** (namespace `AWS/Bedrock`, dimension `ModelId`): `Invocations`,
  `InvocationLatency`, `InputTokenCount`, `OutputTokenCount`, `InvocationThrottles`,
  `InvocationClientErrors`, `InvocationServerErrors`. They exist without any setup.
- **CloudWatch generative AI observability**: ready-made dashboards for *Model Invocations* and for
  *AgentCore agents*, with end-to-end prompt traces. Traces need **Transaction Search** to be enabled.
- **Agent trace** (`enableTrace=True`) shows each step: model call, rationale, tool call, observation,
  failure.
- **Truncated output**: `stopReason == "max_tokens"`, or a stream that ends without `messageStop`.
- **AgentCore Observability** = OpenTelemetry (ADOT) traces, spans, sessions and metrics in CloudWatch.
- **Bedrock evaluations** (automatic, LLM-as-a-judge, human, RAG) measure quality. They are jobs, so
  schedule them or run a small recurring check yourself for continuous monitoring.

| Need | Use |
|---|---|
| Who called which model with which prompt | Model invocation logs (CloudWatch Logs Insights) |
| Latency, token and throttle trends, alarms | `AWS/Bedrock` CloudWatch metrics |
| Why did the agent do that? | Agent trace / AgentCore Observability spans |
| Who changed the configuration? | CloudTrail (not invocation logging) |
| Is answer quality going down? | Bedrock evaluation jobs or a scheduled judge → custom metric |

## Setup

In [ ]:
import sys; sys.path.append("..")
import json, os, time, uuid, datetime
import pandas as pd
from botocore.exceptions import ClientError
from common import utils as U, catchup as C
print(U.REGION, U.bucket())

Catch-up: the Bedrock agent with its claim lookup tool, and the Knowledge Base over the policy
documents (both return at once if an earlier notebook built them).

In [ ]:
AGENT_ID, ALIAS_ID = C.ensure_agent()
KB_ID = U.get_state("kb_id") or C.ensure_kb()
print("agent:", AGENT_ID, ALIAS_ID, "| knowledge base:", KB_ID)

Clients and names.

In [ ]:
bedrock, brt = U.client("bedrock"), U.client("bedrock-runtime")
agent_rt, lam = U.client("bedrock-agent-runtime"), U.client("lambda")
logs, cw, sns, s3 = U.client("logs"), U.client("cloudwatch"), U.client("sns"), U.client("s3")
events, xray = U.client("events"), U.client("xray")

LOG_GROUP = "/safedrive/bedrock/invocations"
LOG_PREFIX = f"{U.PREFIX}/bedrock-logs"
TOOL_LAMBDA = C.TOOL_LAMBDA
pd.set_option("display.max_colwidth", 100)
os.makedirs("scripts", exist_ok=True)          # %%writefile cells write here
print(LOG_GROUP, TOOL_LAMBDA)

## 4.1.1 Model invocation logging

- Logs every `InvokeModel` / `Converse` call (also calls made by agents and Knowledge Bases).
- Destinations: CloudWatch Logs (query with Logs Insights), S3 (cheap, long-term), or both.
- The logs contain prompts and answers, so protect them (KMS, IAM, retention, data masking).

Create the log group with a 30-day retention.

In [ ]:
try:
    logs.create_log_group(logGroupName=LOG_GROUP, tags={t["Key"]: t["Value"] for t in U.TAGS})
except logs.exceptions.ResourceAlreadyExistsException:
    pass
logs.put_retention_policy(logGroupName=LOG_GROUP, retentionInDays=30)
print(logs.describe_log_groups(logGroupNamePrefix=LOG_GROUP)["logGroups"][0]["retentionInDays"], "days retention")

Bedrock writes to S3 as the service principal `bedrock.amazonaws.com`, so the bucket policy must
allow it. This cell adds one statement to the practice bucket policy (existing statements are kept).

In [ ]:
stmt = {"Sid": "AmazonBedrockLogsWrite", "Effect": "Allow", "Principal": {"Service": "bedrock.amazonaws.com"},
        "Action": "s3:PutObject",
        "Resource": f"arn:aws:s3:::{U.bucket()}/{LOG_PREFIX}/AWSLogs/{U.account_id()}/BedrockModelInvocationLogs/*",
        "Condition": {"StringEquals": {"aws:SourceAccount": U.account_id()},
                      "ArnLike": {"aws:SourceArn": f"arn:aws:bedrock:{U.REGION}:{U.account_id()}:*"}}}
try:
    bucket_policy = json.loads(s3.get_bucket_policy(Bucket=U.bucket())["Policy"])
except ClientError:
    bucket_policy = {"Version": "2012-10-17", "Statement": []}
bucket_policy["Statement"] = [s for s in bucket_policy["Statement"] if s.get("Sid") != stmt["Sid"]] + [stmt]
s3.put_bucket_policy(Bucket=U.bucket(), Policy=json.dumps(bucket_policy))
print(len(bucket_policy["Statement"]), "statement(s) in the bucket policy")

Check the current setting first. There is only **one** logging configuration per account and Region,
so we do not overwrite one that somebody else set up.

In [ ]:
current = bedrock.get_model_invocation_logging_configuration().get("loggingConfig")
print(json.dumps(current, indent=1) if current else "invocation logging is OFF")

Turn logging on (CloudWatch Logs + S3, text only). If a configuration already exists we keep it and
query its log group instead.

In [ ]:
if current and current.get("cloudWatchConfig"):
    LOG_GROUP = current["cloudWatchConfig"]["logGroupName"]
    print("keeping the existing configuration, log group:", LOG_GROUP)
else:
    bedrock.put_model_invocation_logging_configuration(loggingConfig={
        "cloudWatchConfig": {"logGroupName": LOG_GROUP, "roleArn": U.role_arn()},
        "s3Config": {"bucketName": U.bucket(), "keyPrefix": LOG_PREFIX},
        "textDataDeliveryEnabled": True, "imageDataDeliveryEnabled": False,
        "embeddingDataDeliveryEnabled": False})
    print("invocation logging is ON ->", LOG_GROUP, "and", U.s3_uri(LOG_PREFIX))

Create some traffic: six questions, sent to the small and the standard model.

In [ ]:
questions = [q["question"] for q in U.read_jsonl(f"{C.RAW}/qa_eval.jsonl")][:6]
for i, q in enumerate(questions):
    model = U.SMALL_MODEL if i % 2 else U.TEXT_MODEL
    r = brt.converse(modelId=model, messages=[{"role": "user", "content": [{"text": q}]}],
                     inferenceConfig={"maxTokens": 150})
    print(model, r["usage"]["inputTokens"], "in /", r["usage"]["outputTokens"], "out /", r["metrics"]["latencyMs"], "ms")

A helper that runs a **CloudWatch Logs Insights** query and returns a DataFrame.

In [ ]:
def insights(log_group, query, minutes=60):
    now = int(time.time())
    qid = logs.start_query(logGroupName=log_group, startTime=now - 60 * minutes, endTime=now,
                           queryString=query)["queryId"]
    while True:
        r = logs.get_query_results(queryId=qid)
        if r["status"] not in ("Scheduled", "Running"):
            break
        time.sleep(2)
    return pd.DataFrame([{c["field"]: c["value"] for c in row if c["field"] != "@ptr"} for row in r["results"]])

print("helper ready")

Query 1: the latest calls. Log delivery takes 1-3 minutes, so wait a little and rerun if it is empty.
Each record has the model, the operation, the caller identity and the token counts.

In [ ]:
time.sleep(90)
insights(LOG_GROUP, """fields @timestamp, modelId, operation, identity.arn, input.inputTokenCount, output.outputTokenCount
| sort @timestamp desc | limit 10""")

Query 2: token use per model. This is the basis for cost reports and for "who uses the most tokens".

In [ ]:
insights(LOG_GROUP, """stats count(*) as calls, sum(input.inputTokenCount) as input_tokens,
  sum(output.outputTokenCount) as output_tokens, avg(output.outputTokenCount) as avg_output by modelId""")

Query 3: find calls by content. Here: every call whose prompt mentions "deductible".

In [ ]:
insights(LOG_GROUP, """fields @timestamp, modelId, @message
| filter @message like /deductible/ | sort @timestamp desc | limit 3""")

The same records also land in S3 as compressed JSON files (delivery takes a few minutes).

In [ ]:
log_keys = U.list_keys(f"{LOG_PREFIX}/AWSLogs/")
print(len(log_keys), "objects")
log_keys[-3:]

### Bedrock runtime metrics and alarms

These metrics need no setup. First list which `ModelId` values have data (for an inference profile
the dimension value is the profile ID you called).

In [ ]:
found = cw.list_metrics(Namespace="AWS/Bedrock", MetricName="Invocations")["Metrics"]
pd.DataFrame([{d["Name"]: d["Value"] for d in m["Dimensions"]} for m in found]).drop_duplicates()

Read the last hour for one model: calls, average latency, tokens and throttles.
Metrics appear 1-2 minutes after the calls.

In [ ]:
def bedrock_metric(metric, stat, model_id, minutes=60):
    now = datetime.datetime.now(datetime.timezone.utc)
    points = cw.get_metric_statistics(
        Namespace="AWS/Bedrock", MetricName=metric, Statistics=[stat], Period=60 * minutes,
        StartTime=now - datetime.timedelta(minutes=minutes), EndTime=now,
        Dimensions=[{"Name": "ModelId", "Value": model_id}])["Datapoints"]
    return round(sum(p[stat] for p in points) / (len(points) if stat == "Average" and points else 1), 1)

wanted = [("Invocations", "Sum"), ("InvocationLatency", "Average"), ("InputTokenCount", "Sum"),
          ("OutputTokenCount", "Sum"), ("InvocationThrottles", "Sum"), ("InvocationClientErrors", "Sum")]
pd.DataFrame({m: {name: bedrock_metric(name, stat, m) for name, stat in wanted}
              for m in [U.TEXT_MODEL, U.SMALL_MODEL]})

Create the alert topic (reused by all alarms in this notebook).

In [ ]:
topic_arn = sns.create_topic(Name="safedrive-genai-alerts",
                             Tags=[{"Key": t["Key"], "Value": t["Value"]} for t in U.TAGS])["TopicArn"]
print(topic_arn)

Alarm on **throttling**: more than 5 throttled calls in 5 minutes means you hit the quota
(ask for a quota increase, use cross-Region inference, or buy provisioned throughput).

In [ ]:
model_dim = [{"Name": "ModelId", "Value": U.TEXT_MODEL}]
cw.put_metric_alarm(
    AlarmName="safedrive-bedrock-throttles", Namespace="AWS/Bedrock", MetricName="InvocationThrottles",
    Dimensions=model_dim, Statistic="Sum", Period=300, EvaluationPeriods=1, Threshold=5,
    ComparisonOperator="GreaterThanThreshold", TreatMissingData="notBreaching",
    AlarmActions=[topic_arn], Tags=U.TAGS)
print("alarm created: safedrive-bedrock-throttles")

Alarm on **latency**: p90 above 5 seconds for 3 periods in a row (`InvocationLatency` is in milliseconds).

In [ ]:
cw.put_metric_alarm(
    AlarmName="safedrive-bedrock-latency", Namespace="AWS/Bedrock", MetricName="InvocationLatency",
    Dimensions=model_dim, ExtendedStatistic="p90", Period=300, EvaluationPeriods=3, Threshold=5000,
    ComparisonOperator="GreaterThanThreshold", TreatMissingData="notBreaching",
    AlarmActions=[topic_arn], Tags=U.TAGS)
print("alarm created: safedrive-bedrock-latency")

### CloudWatch generative AI observability

📝 Open **CloudWatch console → GenAI Observability**. It has two ready-made views:

| View | Shows | Needs |
|---|---|---|
| Model Invocations | Invocation count, token use (input, output, per query), latency (avg, p90, p99), errors, throttles, and a table of logged prompts and answers | Model invocation logging (done above) |
| Bedrock AgentCore | Agents, sessions, traces and spans; metrics for runtime, memory, gateway, tools, identity | Transaction Search + ADOT instrumentation in the agent |

- It builds on normal CloudWatch features: metrics, Logs Insights, alarms, dashboards, data protection.
- Works with Strands, LangChain and LangGraph agents through OpenTelemetry.
- Check the current details in the CloudWatch User Guide ("Generative AI observability").

Check where X-Ray sends trace segments today. `CloudWatchLogs` means Transaction Search is on.

In [ ]:
dest = xray.get_trace_segment_destination()
print("destination:", dest.get("Destination"), "| status:", dest.get("Status"))

🔒 Enable **Transaction Search** (account-wide setting; spans are then stored in the `aws/spans` log
group and billed as CloudWatch Logs ingestion). Step 1 lets X-Ray write to the log groups, step 2
switches the destination, step 3 indexes 1% of spans as trace summaries (the free default).

In [ ]:
if U.expensive("enable CloudWatch Transaction Search for the account"):
    arn = f"arn:aws:logs:{U.REGION}:{U.account_id()}:log-group"
    policy = {"Version": "2012-10-17", "Statement": [{
        "Sid": "TransactionSearchXRayAccess", "Effect": "Allow", "Principal": {"Service": "xray.amazonaws.com"},
        "Action": "logs:PutLogEvents", "Resource": [f"{arn}:aws/spans:*", f"{arn}:/aws/application-signals/data:*"],
        "Condition": {"ArnLike": {"aws:SourceArn": f"arn:aws:xray:{U.REGION}:{U.account_id()}:*"},
                      "StringEquals": {"aws:SourceAccount": U.account_id()}}}]}
    logs.put_resource_policy(policyName="safedrive-transaction-search", policyDocument=json.dumps(policy))
    xray.update_trace_segment_destination(Destination="CloudWatchLogs")
    xray.update_indexing_rule(Name="Default", Rule={"Probabilistic": {"DesiredSamplingPercentage": 1}})
    print(xray.get_trace_segment_destination()["Status"])

## 4.1.5 Monitor agent performance and coordination

- `invoke_agent(enableTrace=True)` returns **trace events** between the answer chunks.
- An orchestration step has: model input/output (with token usage) → rationale → tool call
  (`invocationInput`) → result (`observation`) → ... → final response.
- Problems to detect: **tool failures**, **truncated output**, **loops / coordination failures**.

A helper that calls the agent and collects the answer, the trace events and any stream error.

In [ ]:
def run_agent(text, session_id=None):
    resp = agent_rt.invoke_agent(agentId=AGENT_ID, agentAliasId=ALIAS_ID, inputText=text,
                                 sessionId=session_id or str(uuid.uuid4()), enableTrace=True)
    answer, traces, error = "", [], None
    try:
        for event in resp["completion"]:
            if "chunk" in event:
                answer += event["chunk"]["bytes"].decode()
            elif "trace" in event:
                traces.append(event["trace"])
    except Exception as e:                      # errors can arrive in the middle of the stream
        error = f"{type(e).__name__}: {e}"
    return answer, traces, error

answer, traces, error = run_agent("What is the status of claim C0000001?")
print("answer:", answer, "\nerror:", error, "\ntrace events:", len(traces))

Turn the trace into a table: one row per step, with what happened in it.

In [ ]:
def trace_table(traces):
    rows = []
    for t in traces:
        inner = t["trace"]
        if "failureTrace" in inner:
            rows.append({"step": "FAILURE", "detail": inner["failureTrace"].get("failureReason", "")})
        o = inner.get("orchestrationTrace", {})
        if "modelInvocationOutput" in o:
            u = o["modelInvocationOutput"].get("metadata", {}).get("usage", {})
            rows.append({"step": "model call", "detail": f"{u.get('inputTokens')} in / {u.get('outputTokens')} out tokens"})
        if "rationale" in o:
            rows.append({"step": "rationale", "detail": o["rationale"].get("text", "")})
        if "invocationInput" in o:
            a = o["invocationInput"].get("actionGroupInvocationInput", {})
            params = {p["name"]: p["value"] for p in a.get("parameters", [])}
            rows.append({"step": "tool call", "detail": f"{a.get('function')} {params}"})
        if "observation" in o:
            ob = o["observation"]
            text = ob.get("actionGroupInvocationOutput", ob.get("finalResponse", {})).get("text", "")
            rows.append({"step": f"observation ({ob.get('type')})", "detail": text})
    return pd.DataFrame(rows)

trace_table(traces)

### Tool failures

**Soft failure:** the tool runs but returns nothing useful. Ask for a claim that does not exist and
look at the observation: `"found": false`. The agent should say so and not invent a status.

In [ ]:
answer, traces, error = run_agent("What is the status of claim C9999999?")
print("answer:", answer)
steps = trace_table(traces)
steps[steps["step"].str.contains("observation|tool call")] if len(steps) else steps

**Hard failure:** break the tool on purpose. Point the Lambda function at a table that does not
exist, so every call raises an error.

In [ ]:
lam.update_function_configuration(FunctionName=TOOL_LAMBDA,
                                  Environment={"Variables": {"CLAIMS_TABLE": "safedrive-claims-missing"}})
lam.get_waiter("function_updated_v2").wait(FunctionName=TOOL_LAMBDA)
print(lam.get_function_configuration(FunctionName=TOOL_LAMBDA)["Environment"]["Variables"])

Detection 1: call the tool directly. `FunctionError` in the response means the function crashed.

In [ ]:
r = lam.invoke(FunctionName=TOOL_LAMBDA, Payload=json.dumps({"claim_id": "C0000001"}))
print("FunctionError:", r.get("FunctionError"))
print(r["Payload"].read().decode()[:300])

Detection 2: call the agent. Depending on the error, you get a `failureTrace` step or an exception in
the stream (for example `dependencyFailedException`). Both mean "a tool or dependency failed".

In [ ]:
answer, traces, error = run_agent("What is the status of claim C0000002?")
print("answer:", answer or "(none)")
print("stream error:", error)
trace_table(traces).tail(4)

One function that reports the health of an agent run. Use it in tests and in monitoring code.

In [ ]:
def tool_health(traces, error):
    table = trace_table(traces)
    details = " ".join(table["detail"].astype(str)) if len(table) else ""
    return {"hard_failure": bool(error) or (len(table) > 0 and (table["step"] == "FAILURE").any()),
            "soft_failure": '"found": false' in details,
            "tool_calls": int((table["step"] == "tool call").sum()) if len(table) else 0}

print("broken tool run:", tool_health(traces, error))

Repair the tool (restore the environment variable) and check that the agent works again.

In [ ]:
lam.update_function_configuration(FunctionName=TOOL_LAMBDA,
                                  Environment={"Variables": {"CLAIMS_TABLE": C.CLAIMS_TABLE}})
lam.get_waiter("function_updated_v2").wait(FunctionName=TOOL_LAMBDA)
answer, traces, error = run_agent("What is the status of claim C0000002?")
print(answer, "\n", tool_health(traces, error))

Detection 3 (automatic): a CloudWatch alarm on the Lambda `Errors` metric of the tool.
In production add alarms on `Throttles` and `Duration` too.

In [ ]:
cw.put_metric_alarm(
    AlarmName="safedrive-agent-tool-errors", Namespace="AWS/Lambda", MetricName="Errors",
    Dimensions=[{"Name": "FunctionName", "Value": TOOL_LAMBDA}], Statistic="Sum", Period=60,
    EvaluationPeriods=1, Threshold=1, ComparisonOperator="GreaterThanOrEqualToThreshold",
    TreatMissingData="notBreaching", AlarmActions=[topic_arn], Tags=U.TAGS)
print("alarm created: safedrive-agent-tool-errors")

The errors we just caused are in the tool's Lambda log group. Find them with Logs Insights
(wait a minute and rerun if the result is empty).

In [ ]:
try:
    display(insights(f"/aws/lambda/{TOOL_LAMBDA}",
                     "fields @timestamp, @message | filter @message like /ERROR|Exception/ | sort @timestamp desc | limit 3"))
except ClientError as e:
    print("log group not ready yet:", e.response["Error"]["Code"])

### Truncated output

- `stopReason` tells why the model stopped: `end_turn` (complete), `max_tokens` (**cut off**),
  `stop_sequence`, `tool_use`, `guardrail_intervened`, `content_filtered`.
- Always check it before you show or store an answer.

In [ ]:
msg = [{"role": "user", "content": [{"text": "Explain in detail how a SafeDrive motor claim is processed."}]}]
r = brt.converse(modelId=U.TEXT_MODEL, messages=msg, inferenceConfig={"maxTokens": 40})
print("stopReason:", r["stopReason"], "| output tokens:", r["usage"]["outputTokens"])
print(r["output"]["message"]["content"][0]["text"])

With **streaming**, a complete answer ends with a `messageStop` event (with the stop reason) and a
`metadata` event (with usage). If the stream ends without them, the connection was cut.
`cut_after` simulates a dropped connection.

In [ ]:
def read_stream(stream, cut_after=None):
    text, stop, usage = "", None, None
    for i, ev in enumerate(stream):
        if "contentBlockDelta" in ev:
            text += ev["contentBlockDelta"]["delta"].get("text", "")
        elif "messageStop" in ev:
            stop = ev["messageStop"]["stopReason"]
        elif "metadata" in ev:
            usage = ev["metadata"]["usage"]
        if cut_after and i >= cut_after:
            break
    complete = stop in ("end_turn", "stop_sequence", "tool_use") and usage is not None
    return {"complete": complete, "stop_reason": stop, "has_usage": usage is not None, "chars": len(text)}

for label, max_tokens, cut in [("normal", 300, None), ("max_tokens", 40, None), ("dropped stream", 300, 8)]:
    s = brt.converse_stream(modelId=U.TEXT_MODEL, messages=msg, inferenceConfig={"maxTokens": max_tokens})
    print(label, "->", read_stream(s["stream"], cut))

An automatic response to truncation: ask the model to continue and join the parts
(or retry with a larger `maxTokens`, or with a shorter prompt).

In [ ]:
def complete_answer(messages, max_tokens=120, max_rounds=4):
    messages, parts = list(messages), []
    for _ in range(max_rounds):
        r = brt.converse(modelId=U.TEXT_MODEL, messages=messages, inferenceConfig={"maxTokens": max_tokens})
        parts.append(r["output"]["message"]["content"][0]["text"])
        if r["stopReason"] != "max_tokens":
            break
        messages += [r["output"]["message"],
                     {"role": "user", "content": [{"text": "Continue exactly where you stopped."}]}]
    return " ".join(parts), r["stopReason"], len(parts)

text, stop, rounds = complete_answer(msg)
print(f"rounds: {rounds} | final stopReason: {stop} | {len(text)} characters")

### Coordination failures and loops

📝 What goes wrong when agents call tools or other agents:

| Failure | How it looks | Detect | Respond |
|---|---|---|---|
| Loop | Same tool with same input again and again | Count repeated tool calls in the trace | Max steps / max iterations, stop the session |
| Wrong routing | Supervisor sends the task to the wrong collaborator | `routingClassifierTrace`, `collaboratorName`, `callerChain` in the trace | Better collaborator descriptions and instructions |
| Hand-off never returns | Collaborator or tool hangs | Timeouts, step duration in trace metadata | Client and Lambda timeouts, retry, fallback |
| Lost context | Collaborator lacks information from earlier turns | Inspect collaborator input in the trace | Share conversation history, use session attributes / memory |
| Token blow-up | Many steps, long history | Sum usage over all steps | Summarise history, limit steps, smaller model |

In multi-agent collaboration (supervisor + collaborators) each trace event names the agent that
produced it. The function below measures any trace, single- or multi-agent.

In [ ]:
def coordination_stats(traces, max_steps=6):
    table = trace_table(traces)
    tool_calls = list(table.loc[table["step"] == "tool call", "detail"]) if len(table) else []
    agents = {t.get("collaboratorName") or "main agent" for t in traces}
    model_calls = int((table["step"] == "model call").sum()) if len(table) else 0
    return {"agents_involved": sorted(agents), "model_calls": model_calls, "tool_calls": len(tool_calls),
            "repeated_tool_calls": len(tool_calls) - len(set(tool_calls)),
            "possible_loop": model_calls > max_steps or len(tool_calls) - len(set(tool_calls)) >= 2}

answer, traces, error = run_agent("Compare the status of claims C0000003 and C0000004.")
print(answer)
coordination_stats(traces)

A **client-side timeout** protects the caller when an agent hangs. `read_timeout` stops waiting for
the stream; `max_attempts` controls retries.

In [ ]:
from botocore.config import Config
agent_rt_guarded = U.session.client("bedrock-agent-runtime",
                                    config=Config(read_timeout=60, connect_timeout=5, retries={"max_attempts": 2}))
r = agent_rt_guarded.invoke_agent(agentId=AGENT_ID, agentAliasId=ALIAS_ID, sessionId=str(uuid.uuid4()),
                                  inputText="Status of claim C0000005?")
print("".join(e["chunk"]["bytes"].decode() for e in r["completion"] if "chunk" in e))

### Automate the response

- **Retries with backoff** for throttling and timeouts: built into the SDK (`standard` / `adaptive` mode).
- **Fallback model** when the first model fails or is not available.
- **Alarm → SNS → Lambda / on-call** for anything that needs a person or a config change.

In [ ]:
brt_retry = U.session.client("bedrock-runtime", config=Config(retries={"max_attempts": 5, "mode": "adaptive"}))

def converse_with_fallback(prompt, models):
    for model in models:
        try:
            r = brt_retry.converse(modelId=model, messages=[{"role": "user", "content": [{"text": prompt}]}],
                                   inferenceConfig={"maxTokens": 100})
            return model, r["output"]["message"]["content"][0]["text"]
        except ClientError as e:
            print(f"{model} failed with {e.response['Error']['Code']} -> trying the next model")
    raise RuntimeError("all models failed")

# The first model ID does not exist, so the call falls back to the second one.
print(converse_with_fallback("In one sentence: what is a deductible?",
                             ["us.amazon.nova-not-a-real-model-v1:0", U.SMALL_MODEL]))

Bedrock Agents also publish their own CloudWatch metrics. List what your account has
(namespace `AWS/Bedrock/Agents`; it can be empty until a few minutes after the first calls).

In [ ]:
agent_metrics = cw.list_metrics(Namespace="AWS/Bedrock/Agents")["Metrics"]
print(sorted({m["MetricName"] for m in agent_metrics}) or "no agent metrics yet")

## 4.2.2 AgentCore Observability and AWS X-Ray

- **AgentCore Runtime** gives built-in metrics (invocations, latency, errors, session count) and a
  log group `/aws/bedrock-agentcore/runtimes/<agent_id>-<endpoint_name>` without extra work.
- For **traces and spans** of your own agent code: add `aws-opentelemetry-distro` (ADOT) to the agent
  and start it with `opentelemetry-instrument python my_agent.py`. Strands, LangChain and CrewAI emit
  OpenTelemetry GenAI spans (model calls, tool calls) on their own.
- One-time account setup: **Transaction Search** (see the 🔒 cell above).
- Hierarchy: **session** (one conversation) → **trace** (one request) → **span** (one step: model
  call, tool call, memory read).
- Send the session ID in the header `X-Amzn-Bedrock-AgentCore-Runtime-Session-Id` (or
  `runtimeSessionId` in the SDK) so that spans are grouped by session.
- View them in the CloudWatch GenAI Observability console (Bedrock AgentCore tab) or query the logs.
- Agents that run outside AgentCore (Lambda, ECS, EKS) can send the same telemetry with ADOT.

If notebook 3.1b deployed the Strands agent to AgentCore Runtime, call it once so that fresh
telemetry exists. Without it, this cell only prints a note.

In [ ]:
runtime_arn = U.get_state("agentcore_runtime_arn")
if runtime_arn:
    ac = U.client("bedrock-agentcore")
    r = ac.invoke_agent_runtime(agentRuntimeArn=runtime_arn, runtimeSessionId=f"safedrive-obs-{uuid.uuid4()}",
                                payload=json.dumps({"prompt": "What is the status of claim C0000001?"}).encode())
    print(r["response"].read().decode()[:400])
else:
    print("No AgentCore runtime in the state file (notebook 3.1b creates it). Read the notes and continue.")

List the AgentCore log groups and the CloudWatch metrics of the runtime.

In [ ]:
groups = logs.describe_log_groups(logGroupNamePrefix="/aws/bedrock-agentcore/")["logGroups"]
print("log groups:", [g["logGroupName"] for g in groups] or "none")
for ns in ["AWS/Bedrock-AgentCore", "bedrock-agentcore"]:
    names = sorted({m["MetricName"] for m in cw.list_metrics(Namespace=ns)["Metrics"]})
    print(ns, "->", names[:12] or "no metrics")

Read the newest runtime log lines (agent output and errors). Skipped if there is no log group.

In [ ]:
if groups:
    display(insights(groups[0]["logGroupName"], "fields @timestamp, @message | sort @timestamp desc | limit 5"))
else:
    print("no AgentCore log group")

With Transaction Search on, spans are log events in the `aws/spans` log group (or in the agent's own
log group, stream `spans`). Query them like any other log.

In [ ]:
try:
    display(insights("aws/spans", "fields @timestamp, name, traceId, durationNano | sort @timestamp desc | limit 5"))
except ClientError as e:
    print("aws/spans not available (Transaction Search is off):", e.response["Error"]["Code"])

**AWS X-Ray basics:** a *trace* is one request across services; it is made of *segments* (one per
service) and *subsegments*. Look up recent traces and their error flags. An empty list is normal if
nothing in the account is instrumented.

In [ ]:
now = datetime.datetime.now(datetime.timezone.utc)
summaries = xray.get_trace_summaries(StartTime=now - datetime.timedelta(minutes=30), EndTime=now)["TraceSummaries"]
print(len(summaries), "traces in the last 30 minutes")
pd.DataFrame([{"id": t["Id"], "duration_s": t.get("Duration"), "error": t.get("HasError"),
               "fault": t.get("HasFault"), "throttle": t.get("HasThrottle")} for t in summaries[:10]])

Open one trace: each segment is a JSON document with a name, start and end time.

In [ ]:
if summaries:
    segments = xray.batch_get_traces(TraceIds=[summaries[0]["Id"]])["Traces"][0]["Segments"]
    docs = [json.loads(s["Document"]) for s in segments]
    display(pd.DataFrame([{"name": d.get("name"), "seconds": round(d.get("end_time", 0) - d.get("start_time", 0), 3),
                           "error": d.get("error", False), "fault": d.get("fault", False)} for d in docs]))
else:
    print("no traces to open")

## 4.1.6 Performance monitoring for foundation models

- Operational metrics (latency, tokens, errors) do not tell you if answers are still **correct**.
- Quality monitoring = run a **fixed set of golden questions** on a schedule, score the answers
  (LLM-as-a-judge or metrics such as ROUGE / BERTScore) and publish the score as a **custom metric**.
- **Amazon Bedrock evaluations** do the scoring as a managed job: automatic (built-in metrics),
  **LLM-as-a-judge** (correctness, completeness, faithfulness, harmfulness ...), human, and **RAG
  evaluation** for Knowledge Bases.
- Things that change quality without a code change: new model version, changed prompt, changed
  documents in the Knowledge Base.

The check as a small script. It works locally and as a Lambda function. Part 1: answer a question
with the Knowledge Base, and let a judge model compare the answer with the reference.

In [ ]:
%%writefile scripts/quality_check.py
"""Recurring quality check for the SafeDrive claims assistant (runs locally or in Lambda)."""
import json
import os

import boto3

REGION = os.environ.get("AWS_REGION")
s3 = boto3.client("s3", region_name=REGION)
brt = boto3.client("bedrock-runtime", region_name=REGION)
kb = boto3.client("bedrock-agent-runtime", region_name=REGION)
cw = boto3.client("cloudwatch", region_name=REGION)


def answer(question):
    config = {"type": "KNOWLEDGE_BASE", "knowledgeBaseConfiguration": {
        "knowledgeBaseId": os.environ["KB_ID"], "modelArn": os.environ["MODEL_ARN"]}}
    resp = kb.retrieve_and_generate(input={"text": question}, retrieveAndGenerateConfiguration=config)
    return resp["output"]["text"]


def judge(question, reference, candidate):
    prompt = (f"Question: {question}\nReference answer: {reference}\nCandidate answer: {candidate}\n"
              "Does the candidate state the same facts as the reference? Reply with one word: CORRECT or WRONG.")
    resp = brt.converse(modelId=os.environ["JUDGE_MODEL"], inferenceConfig={"maxTokens": 5, "temperature": 0},
                        messages=[{"role": "user", "content": [{"text": prompt}]}])
    return 1.0 if resp["output"]["message"]["content"][0]["text"].strip().upper().startswith("CORRECT") else 0.0

Part 2: the handler reads the golden questions from S3, scores them and publishes the metric
`AnswerCorrectness` (0 to 1) in the namespace `SafeDrive/GenAI`.

In [ ]:
%%writefile -a scripts/quality_check.py


def handler(event, context):
    body = s3.get_object(Bucket=os.environ["BUCKET"], Key=os.environ["EVAL_KEY"])["Body"].read().decode()
    rows = [json.loads(line) for line in body.splitlines() if line.strip()]
    rows = rows[: int(os.environ.get("MAX_QUESTIONS", "8"))]
    scores = [judge(r["question"], r["answer"], answer(r["question"])) for r in rows]
    value = sum(scores) / len(scores)
    cw.put_metric_data(Namespace="SafeDrive/GenAI", MetricData=[{
        "MetricName": "AnswerCorrectness", "Value": value, "Unit": "None",
        "Dimensions": [{"Name": "Application", "Value": "claims-assistant"}]}])
    return {"questions": len(scores), "correctness": value, "scores": scores}

Run the check **locally** first. The settings are environment variables, the same ones the Lambda
function gets. Look at `correctness`: the share of golden questions answered correctly.

In [ ]:
check_env = {"KB_ID": KB_ID, "MODEL_ARN": U.model_arn(U.TEXT_MODEL), "JUDGE_MODEL": U.LARGE_MODEL,
             "BUCKET": U.bucket(), "EVAL_KEY": f"{C.RAW}/qa_eval.jsonl", "MAX_QUESTIONS": "8"}
os.environ.update(check_env)
os.environ.setdefault("AWS_REGION", U.REGION)
sys.path.insert(0, "scripts")
import quality_check
result = quality_check.handler({}, None)
result

Deploy the same file as a Lambda function (5 minute timeout, because it makes many model calls).

In [ ]:
import io, zipfile
buf = io.BytesIO()
with zipfile.ZipFile(buf, "w") as z:
    z.write("scripts/quality_check.py", "quality_check.py")
CHECK_FN = "safedrive-quality-check"
try:
    lam.create_function(FunctionName=CHECK_FN, Runtime="python3.12", Role=U.role_arn(),
                        Handler="quality_check.handler", Code={"ZipFile": buf.getvalue()}, Timeout=300,
                        MemorySize=256, Environment={"Variables": check_env},
                        Tags={t["Key"]: t["Value"] for t in U.TAGS})
except lam.exceptions.ResourceConflictException:
    lam.update_function_code(FunctionName=CHECK_FN, ZipFile=buf.getvalue())
lam.get_waiter("function_active_v2").wait(FunctionName=CHECK_FN)
print(lam.get_function_configuration(FunctionName=CHECK_FN)["State"])

Invoke the function once to test it (takes about a minute). The result should look like the local run.

In [ ]:
lam_long = U.session.client("lambda", config=Config(read_timeout=320, retries={"max_attempts": 0}))
r = lam_long.invoke(FunctionName=CHECK_FN, Payload=b"{}")
print("FunctionError:", r.get("FunctionError"))
print(r["Payload"].read().decode()[:400])

Make it **recurring**: an EventBridge schedule rule runs the function once a day.

In [ ]:
rule_arn = events.put_rule(Name="safedrive-quality-check-daily", ScheduleExpression="rate(1 day)",
                           State="ENABLED", Description="Daily golden-question check", Tags=U.TAGS)["RuleArn"]
fn_arn = lam.get_function_configuration(FunctionName=CHECK_FN)["FunctionArn"]
try:
    lam.add_permission(FunctionName=CHECK_FN, StatementId="eventbridge-daily", Action="lambda:InvokeFunction",
                       Principal="events.amazonaws.com", SourceArn=rule_arn)
except lam.exceptions.ResourceConflictException:
    pass
events.put_targets(Rule="safedrive-quality-check-daily", Targets=[{"Id": "check", "Arn": fn_arn}])
print(rule_arn)

Alarm when correctness falls below 0.8. Missing data is treated as **breaching**: if the check
itself stops running, you want to know.

In [ ]:
cw.put_metric_alarm(
    AlarmName="safedrive-answer-correctness-low", Namespace="SafeDrive/GenAI", MetricName="AnswerCorrectness",
    Dimensions=[{"Name": "Application", "Value": "claims-assistant"}], Statistic="Average", Period=86400,
    EvaluationPeriods=1, Threshold=0.8, ComparisonOperator="LessThanThreshold",
    TreatMissingData="breaching", AlarmActions=[topic_arn], Tags=U.TAGS)
print("alarm created: safedrive-answer-correctness-low")

Read the custom metric back (the two runs above; wait a minute and rerun if it is empty).

In [ ]:
now = datetime.datetime.now(datetime.timezone.utc)
points = cw.get_metric_statistics(
    Namespace="SafeDrive/GenAI", MetricName="AnswerCorrectness", Statistics=["Average", "SampleCount"],
    Dimensions=[{"Name": "Application", "Value": "claims-assistant"}], Period=3600,
    StartTime=now - datetime.timedelta(hours=3), EndTime=now)["Datapoints"]
pd.DataFrame(points)

### Amazon Bedrock evaluation job

🔒 The managed way: an **LLM-as-a-judge** model evaluation job. Cost: model inference for every
prompt plus judge-model calls; the job takes 15-30 minutes. The dataset is JSONL with `prompt` and
`referenceResponse`. First write the dataset (cheap, always runs).

In [ ]:
rows = U.read_jsonl(f"{C.RAW}/qa_eval.jsonl")
eval_lines = "\n".join(json.dumps({"prompt": r["question"], "referenceResponse": r["answer"]}) for r in rows)
EVAL_KEY = f"{U.PREFIX}/evaluation/qa_eval_bedrock.jsonl"
s3.put_object(Bucket=U.bucket(), Key=EVAL_KEY, Body=eval_lines.encode())
print(len(rows), "prompts ->", U.s3_uri(EVAL_KEY))

🔒 Start the evaluation job. Results (scores per metric and per prompt, with the judge's explanation)
go to S3 and are shown in the Bedrock console under **Evaluations**.

In [ ]:
if U.expensive("Bedrock LLM-as-a-judge evaluation job"):
    job = bedrock.create_evaluation_job(
        jobName=f"safedrive-qa-eval-{int(time.time())}", roleArn=U.role_arn(), applicationType="ModelEvaluation",
        evaluationConfig={"automated": {
            "datasetMetricConfigs": [{
                "taskType": "General",       # LLM-as-a-judge jobs use the task type "General"
                "dataset": {"name": "safedrive-qa", "datasetLocation": {"s3Uri": U.s3_uri(EVAL_KEY)}},
                "metricNames": ["Builtin.Correctness", "Builtin.Completeness", "Builtin.Helpfulness"]}],
            "evaluatorModelConfig": {"bedrockEvaluatorModels": [{"modelIdentifier": U.LARGE_MODEL}]}}},
        inferenceConfig={"models": [{"bedrockModel": {
            "modelIdentifier": U.TEXT_MODEL,
            "inferenceParams": json.dumps({"inferenceConfig": {"maxTokens": 300, "temperature": 0}})}}]},
        outputDataConfig={"s3Uri": U.s3_uri(f"{U.PREFIX}/evaluation/results/")})
    U.save_state(evaluation_job_arn=job["jobArn"])
    print(job["jobArn"])

🔒 Check the job status (rerun until `Completed`). To monitor continuously, start a job like this
from a schedule (EventBridge → Lambda) and alarm on the scores, as we did with the small check.

In [ ]:
if U.expensive("Bedrock evaluation job status"):
    j = bedrock.get_evaluation_job(jobIdentifier=U.get_state("evaluation_job_arn"))
    print(j["status"], j.get("failureMessages", ""))
    print(U.list_keys(f"{U.PREFIX}/evaluation/results/")[:5])

📝 Other evaluation options to know:
- **RAG evaluation job** (`applicationType="RagEvaluation"`): scores a Knowledge Base on retrieval
  (context relevance, coverage) and generation (correctness, faithfulness, citation quality).
- **Automatic (programmatic) evaluation**: built-in datasets and metrics (accuracy, robustness, toxicity).
- **Human evaluation**: your own team or an AWS-managed team rates the answers.
- **Guardrail metrics** in CloudWatch (`AWS/Bedrock/Guardrails`) show how often answers are blocked.

## Exam traps

- Model invocation logging records **prompts and answers**; CloudTrail records **API calls** (who,
  when), not the content. They are different tools.
- Invocation logging is **per Region** and **off by default**; metrics in `AWS/Bedrock` are always on.
- `stopReason: max_tokens` is not an error: the call succeeds with HTTP 200. You must check it yourself.
- Agent traces are returned only when `enableTrace=True`; AgentCore spans need **Transaction Search**
  and **ADOT** instrumentation.
- An evaluation job is a one-time batch job. Continuous quality monitoring needs a schedule.

## Quiz

**1.** Compliance asks for the full prompt and response of every Bedrock call, kept for 7 years at
the lowest cost. What do you configure?
A) CloudTrail data events  B) Model invocation logging to S3 with a lifecycle rule
C) CloudWatch metrics with 7-year retention  D) X-Ray tracing

<details><summary>Answer</summary>B. Invocation logging stores request and response bodies; S3 with lifecycle rules is the cheapest long-term store.</details>

**2.** Users report that long answers of the assistant end in the middle of a sentence. No errors are
logged. What should the application check?
A) `InvocationThrottles`  B) The `stopReason` field for `max_tokens`
C) The guardrail trace  D) The Lambda `Errors` metric

<details><summary>Answer</summary>B. A truncated answer is a successful call with stopReason max_tokens; raise maxTokens or continue the answer.</details>

**3.** A Bedrock agent sometimes answers "I could not find the claim" for valid claim IDs. How do you
find out which step fails?
A) Enable the agent trace and look at the tool invocation and observation steps
B) Increase the model temperature  C) Read the CloudTrail event history  D) Create a new agent version

<details><summary>Answer</summary>A. The trace shows the tool input, the tool result and any failureTrace for each step.</details>

**4.** A Strands agent runs on AgentCore Runtime. The team wants to see sessions, traces and spans in
CloudWatch. What is required?
A) Model invocation logging only  B) Transaction Search enabled and the agent instrumented with ADOT (OpenTelemetry)
C) A CloudTrail trail  D) An AWS Config rule

<details><summary>Answer</summary>B. AgentCore Observability uses OpenTelemetry spans delivered through Transaction Search.</details>

**5.** After a document update in the Knowledge Base, answer quality may drop without any error.
Which solution detects this automatically?
A) Alarm on `InvocationLatency`  B) Scheduled golden-question evaluation with a judge model that publishes a score metric with an alarm
C) Higher `maxTokens`  D) Provisioned throughput

<details><summary>Answer</summary>B. Only a quality evaluation measures correctness; schedule it and alarm on the score.</details>

## Save state

In [ ]:
U.save_state(bedrock_invocation_log_group=LOG_GROUP)
print("saved")

## Cleanup

Kept for later notebooks: the agent, its tool Lambda, the Knowledge Base, and the invocation log group
with its S3 logs (notebook 4.2 reads them). The bucket policy statement for Bedrock logs also stays.
Everything else is deleted below.

Delete the daily schedule rule.

In [ ]:
U.safe(events.remove_targets, Rule="safedrive-quality-check-daily", Ids=["check"])
U.safe(events.delete_rule, Name="safedrive-quality-check-daily")

Delete the quality-check Lambda function.

In [ ]:
U.safe(lam.delete_function, FunctionName="safedrive-quality-check")

Delete the alarms.

In [ ]:
U.safe(cw.delete_alarms, AlarmNames=["safedrive-bedrock-throttles", "safedrive-bedrock-latency",
                                     "safedrive-agent-tool-errors", "safedrive-answer-correctness-low"])

Delete the SNS topic.

In [ ]:
U.safe(sns.delete_topic, TopicArn=f"arn:aws:sns:{U.REGION}:{U.account_id()}:safedrive-genai-alerts")

Optional: turn invocation logging off. Set `TURN_LOGGING_OFF = True` only if **this notebook** turned
it on and you do not plan to run notebook 4.2 next.

In [ ]:
TURN_LOGGING_OFF = False
if TURN_LOGGING_OFF and LOG_GROUP == "/safedrive/bedrock/invocations":
    U.safe(bedrock.delete_model_invocation_logging_configuration)
    U.safe(logs.delete_log_group, logGroupName=LOG_GROUP)
print("invocation logging:", "deleted" if TURN_LOGGING_OFF else "kept")